# Revenue Efficiency Optimization: reproducible analysis

Companion notebook to the project [README](../README.md). Every number quoted in the README is produced by a cell below, so each claim can be checked. Run the cells top to bottom.

**Conventions**

- *Revenue* is `revenue_usd` (net of discount). *List revenue* is `unit_price_usd × units_sold`.
- *Return rate* counts `return_status == "Returned"` only. Exchanges are reported separately.
- Revenue per transaction is strongly right-skewed, so group comparisons use rank-based tests (Kruskal-Wallis, Mann-Whitney) next to means and medians.
- Significance level is 0.05, with no multiple-testing correction. Results between 0.05 and 0.10 are treated as inconclusive.

**Sections**

1. Data quality
2. Headline metrics
3. Concentration (category, product, order size)
4. Geography
5. Discounts
6. Sales channel
7. Returns
8. Trend and seasonality
9. Customer dimensions

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.width", 140)

# Works whether the notebook is opened from the repository root or from analysis/
DATA = next(p for p in (Path("apple_global_sales_dataset.csv"),
                        Path("../apple_global_sales_dataset.csv")) if p.exists())

df = pd.read_csv(DATA, parse_dates=["sale_date"])
RAW_SHAPE = df.shape
REV = df["revenue_usd"].sum()

# Derived fields used throughout
df["list_rev"] = df["unit_price_usd"] * df["units_sold"]
df["disc_usd"] = df["list_rev"] - df["revenue_usd"]
df["returned"] = df["return_status"].eq("Returned")
df["ym"] = df["sale_date"].dt.to_period("M")


def kruskal(col, by):
    """Kruskal-Wallis p-value of `col` across the groups of `by`."""
    return stats.kruskal(*[g[col].dropna().values for _, g in df.groupby(by, observed=True)]).pvalue


print(f"Loaded {RAW_SHAPE[0]:,} rows x {RAW_SHAPE[1]} columns from {DATA}")

Loaded 11,500 rows x 27 columns from ../apple_global_sales_dataset.csv


## 1. Data quality

Before any analysis: are the rows unique, do the money fields reconcile, and are the nulls explainable?

In [2]:
price_diff = (df.unit_price_usd * (1 - df.discount_pct / 100) - df.discounted_price_usd).abs().max()
rev_diff = (df.discounted_price_usd * df.units_sold - df.revenue_usd).abs().max()
fx_diff = ((df.revenue_usd * df.fx_rate_to_usd - df.revenue_local_currency).abs()
           / df.revenue_local_currency).max()
calendar_ok = ((df.sale_date.dt.year == df.year)
               & (("Q" + df.sale_date.dt.quarter.astype(str)) == df.quarter)
               & (df.sale_date.dt.month_name() == df.month)).all()

checks = pd.DataFrame({
    "check": ["Rows x columns", "Date range", "Duplicate rows", "Duplicate sale_id",
              "Max |unit_price x (1 - discount) - discounted_price|",
              "Max |discounted_price x units - revenue|",
              "Max relative |revenue_usd x fx_rate - revenue_local|",
              "year, quarter, month agree with sale_date"],
    "result": [f"{RAW_SHAPE[0]:,} x {RAW_SHAPE[1]}",
               f"{df.sale_date.min().date()} to {df.sale_date.max().date()}",
               df.duplicated().sum(), df.sale_id.duplicated().sum(),
               f"{price_diff:.3f}", f"{rev_diff:.2f}", f"{fx_diff:.6f} ({fx_diff * 100:.2f}%)",
               calendar_ok],
})
checks

,check,result
0,Rows x columns,"11,500 x 27"
1,Date range,2022-01-01 to 2024-12-31
2,Duplicate rows,0
3,Duplicate sale_id,0
4,Max |unit_price x (1 - discount) - discounted_...,0.005
5,Max |discounted_price x units - revenue|,0.00
6,Max relative |revenue_usd x fx_rate - revenue_...,0.000205 (0.02%)
7,"year, quarter, month agree with sale_date",True


In [3]:
null_share = (df.isna().mean() * 100).round(1)
print("Share of nulls by column (%), only columns with nulls:")
print(null_share[null_share > 0].to_string())

print("\nstorage null share by category:")
print(df.groupby("category")["storage"].apply(lambda s: s.isna().mean()).round(2).to_dict())

print("\nprevious_device_os null share by category:")
print(df.groupby("category")["previous_device_os"].apply(lambda s: s.isna().mean()).round(2).to_dict())

print("\ncustomer_rating null share by return_status:")
print(df.groupby("return_status")["customer_rating"].apply(lambda s: s.isna().mean()).round(3).to_dict())

print("\nDistinct discount_pct values:", [int(x) for x in sorted(df.discount_pct.unique())])
print("Countries labeled 'Europe/Asia':", sorted(df.loc[df.region == "Europe/Asia", "country"].unique()))

Share of nulls by column (%), only columns with nulls:
storage              41.80
previous_device_os   70.10
customer_rating      29.20

storage null share by category:
{'Accessories': 1.0, 'AirPods': 1.0, 'Apple Watch': 1.0, 'Mac': 0.0, 'iPad': 0.0, 'iPhone': 0.0}

previous_device_os null share by category:
{'Accessories': 1.0, 'AirPods': 1.0, 'Apple Watch': 1.0, 'Mac': 1.0, 'iPad': 1.0, 'iPhone': 0.0}

customer_rating null share by return_status:
{'Exchanged': 0.288, 'Kept': 0.294, 'Returned': 0.274}

Distinct discount_pct values: [0, 2, 3, 5, 7, 10, 15]
Countries labeled 'Europe/Asia': ['Russia', 'Turkey']


**Takeaway.** No duplicates, and prices, revenue and FX all reconcile to rounding. The nulls are structural (`storage` exists only for Mac, iPad and iPhone; `previous_device_os` only for iPhone). Missing ratings (29.2%) do not vary with return status, so they are analyzed on the available rows. `discount_pct` has only seven values, which is why discounts are grouped into buckets.

## 2. Headline metrics

In [4]:
headline = pd.Series({
    "Transactions": f"{len(df):,}",
    "Net revenue": f"${REV / 1e6:.2f}M",
    "Units sold": f"{df.units_sold.sum():,}",
    "Revenue per transaction, mean": f"${df.revenue_usd.mean():,.0f}",
    "Revenue per transaction, median": f"${df.revenue_usd.median():,.0f}",
    "Skewness of revenue per transaction": f"{df.revenue_usd.skew():.1f}",
    "Countries / products / categories / channels":
        f"{df.country.nunique()} / {df.product_name.nunique()} / {df.category.nunique()} / {df.sales_channel.nunique()}",
    "Transactions with a discount": f"{(df.discount_pct > 0).mean() * 100:.1f}%",
    "Average discount": f"{df.discount_pct.mean():.2f}%",
})
headline.to_frame("value")

,value
Transactions,"11,500"
Net revenue,$18.04M
Units sold,"23,270"
"Revenue per transaction, mean","$1,568"
"Revenue per transaction, median",$833
Skewness of revenue per transaction,7.2
Countries / products / categories / channels,47 / 43 / 6 / 6
Transactions with a discount,54.6%
Average discount,3.84%


**Takeaway.** The mean ($1,568) is almost twice the median ($833) and skewness is 7.2, so means are dominated by a few large orders. This is why the analysis pairs means with medians and uses rank-based tests.

## 3. Concentration

Where does revenue come from: which categories, which products, and how much from the largest orders?

In [5]:
cat = df.groupby("category").agg(revenue=("revenue_usd", "sum"), transactions=("sale_id", "count"),
                                 revenue_per_txn=("revenue_usd", "mean"))
cat["revenue_share_%"] = cat.revenue / REV * 100
cat["transaction_share_%"] = cat.transactions / len(df) * 100
cat = cat.sort_values("revenue", ascending=False)
display(cat.round(1))

top3 = cat.head(3)
print(f"Mac + iPhone + iPad: {top3['revenue_share_%'].sum():.1f}% of revenue "
      f"from {top3['transaction_share_%'].sum():.1f}% of transactions")

,revenue,transactions,revenue_per_txn,revenue_share_%,transaction_share_%
category,,,,,
Mac,"8,369,961.40",1873,"4,468.70",46.40,16.30
iPhone,"5,734,154.30",3444,"1,665.00",31.80,29.90
iPad,"1,829,815.70",1379,"1,326.90",10.10,12.00
Apple Watch,"958,773.70",1126,851.50,5.30,9.80
AirPods,"572,774.00",1063,538.80,3.20,9.20
Accessories,"570,190.10",2615,218.00,3.20,22.70


Mac + iPhone + iPad: 88.3% of revenue from 58.2% of transactions


In [6]:
prod = df.groupby("product_name")["revenue_usd"].sum().sort_values(ascending=False)
mac_pro = df[df.product_name == "Mac Pro (M2 Ultra)"]
print(f"Mac Pro (M2 Ultra): {mac_pro.revenue_usd.sum() / REV * 100:.1f}% of revenue "
      f"from {len(mac_pro) / len(df) * 100:.1f}% of transactions")
print(f"Top 10 products: {prod.head(10).sum() / REV * 100:.1f}% of revenue")
print(f"Products needed for 80% of revenue: {(prod.cumsum() / REV < 0.8).sum() + 1} of {len(prod)}")

ordered = df.revenue_usd.sort_values(ascending=False)
for pct in (1, 5, 10):
    share = ordered.head(len(df) * pct // 100).sum() / REV * 100
    print(f"Top {pct}% of transactions = {share:.1f}% of revenue")

Mac Pro (M2 Ultra): 20.6% of revenue from 2.4% of transactions
Top 10 products: 57.2% of revenue
Products needed for 80% of revenue: 20 of 43
Top 1% of transactions = 14.3% of revenue
Top 5% of transactions = 33.6% of revenue
Top 10% of transactions = 47.1% of revenue


In [7]:
monthly_total = df.groupby("ym")["revenue_usd"].sum()
monthly_count = df.groupby("ym").size()
monthly_mac = (df[df.category == "Mac"].groupby("ym")["revenue_usd"].sum()
               .reindex(monthly_total.index, fill_value=0))
monthly_mac_pro = (mac_pro.groupby("ym")["revenue_usd"].sum()
                   .reindex(monthly_total.index, fill_value=0))

print(f"Monthly revenue CV: {monthly_total.std() / monthly_total.mean() * 100:.1f}%")
print(f"Monthly transaction-count CV: {monthly_count.std() / monthly_count.mean() * 100:.1f}%")
print(f"Share of monthly revenue variance explained by Mac: "
      f"{np.corrcoef(monthly_total, monthly_mac)[0, 1] ** 2 * 100:.0f}%")
print(f"Share of monthly revenue variance explained by Mac Pro: "
      f"{np.corrcoef(monthly_total, monthly_mac_pro)[0, 1] ** 2 * 100:.0f}%")

Monthly revenue CV: 10.9%
Monthly transaction-count CV: 6.2%
Share of monthly revenue variance explained by Mac: 81%
Share of monthly revenue variance explained by Mac Pro: 57%


**Takeaway.** Three categories produce 88% of revenue from 58% of transactions, and one product (Mac Pro M2 Ultra) produces 20.6% from 2.4%. Monthly revenue is roughly twice as variable as volume, and Mac alone explains 81% of that variation.

## 4. Geography

Regional revenue looks concentrated, but regions cover very different numbers of countries. Revenue per country is the fairer comparison.

In [8]:
geo = df.groupby("region").agg(revenue=("revenue_usd", "sum"), transactions=("sale_id", "count"),
                               countries=("country", "nunique"), revenue_per_txn=("revenue_usd", "mean"))
geo["revenue_share_%"] = geo.revenue / REV * 100
geo["revenue_per_country_$M"] = geo.revenue / geo.countries / 1e6
geo = geo.sort_values("revenue", ascending=False)
display(geo.round(2))

europe_asia = geo.loc[["Europe", "Asia"]]
print(f"Europe + Asia: {europe_asia['revenue_share_%'].sum():.1f}% of revenue, "
      f"{europe_asia.countries.sum()} of {df.country.nunique()} countries")

per_country = df.groupby("country").size()
print(f"Transactions per country: min {per_country.min()}, median {per_country.median():.0f}, max {per_country.max()}")
print(f"Kruskal-Wallis, revenue per transaction ~ region: p = {kruskal('revenue_usd', 'region'):.3f}")

,revenue,transactions,countries,revenue_per_txn,revenue_share_%,revenue_per_country_$M
region,,,,,,
Europe,"6,210,935.78",3898,16,"1,593.36",34.44,0.39
Asia,"5,432,379.44",3435,14,"1,581.48",30.12,0.39
South America,"1,418,014.98",990,4,"1,432.34",7.86,0.35
Africa,"1,379,004.44",943,4,"1,462.36",7.65,0.34
North America,"1,186,765.39",762,3,"1,557.43",6.58,0.40
Middle East,"814,583.85",486,2,"1,676.10",4.52,0.41
Europe/Asia,"800,714.79",480,2,"1,668.16",4.44,0.40
Oceania,"793,270.58",506,2,"1,567.73",4.40,0.40


Europe + Asia: 64.6% of revenue, 30 of 47 countries
Transactions per country: min 208, median 244, max 274
Kruskal-Wallis, revenue per transaction ~ region: p = 0.474


**Takeaway.** Europe and Asia hold 65% of revenue because they cover 30 of the 47 countries. Revenue per country is $0.34M to $0.41M in every region, and transactions per country range only from 208 to 274.

## 5. Discounts

What do discounts cost, and is there any evidence that they lift basket size?

In [9]:
df["bucket"] = pd.cut(df.discount_pct, [-1, 0, 5, 10, 15],
                      labels=["No Discount", "Low (1-5%)", "Medium (6-10%)", "High (11-15%)"])

disc = df.groupby("bucket", observed=True).agg(
    transactions=("sale_id", "count"), net_mean=("revenue_usd", "mean"), net_median=("revenue_usd", "median"),
    list_mean=("list_rev", "mean"), units_per_txn=("units_sold", "mean"), discount_usd=("disc_usd", "sum"))
disc["transaction_share_%"] = disc.transactions / len(df) * 100
disc["discount_dollar_share_%"] = disc.discount_usd.clip(lower=0) / disc.discount_usd.clip(lower=0).sum() * 100
display(disc.round(1) + 0.0)

none, high = disc.loc["No Discount"], disc.loc["High (11-15%)"]
print(f"High vs No Discount: mean net revenue per transaction {high.net_mean / none.net_mean * 100 - 100:.1f}%, "
      f"median {high.net_median / none.net_median * 100 - 100:.1f}%")
print(f"Discount given up: ${df.disc_usd.sum() / 1e6:.2f}M = {df.disc_usd.sum() / df.list_rev.sum() * 100:.2f}% of list revenue")
print(f"Transactions with a discount: {(df.discount_pct > 0).mean() * 100:.1f}%")

,transactions,net_mean,net_median,list_mean,units_per_txn,discount_usd,transaction_share_%,discount_dollar_share_%
bucket,,,,,,,,
No Discount,"5,226.00","1,628.80",865.70,"1,628.80",2.00,0.00,45.40,0.00
Low (1-5%),"3,082.00","1,620.00",834.20,"1,676.50",2.00,"174,148.10",26.80,24.90
Medium (6-10%),"2,155.00","1,524.30",786.50,"1,666.00",2.00,"305,378.30",18.70,43.70
High (11-15%),"1,037.00","1,201.50",706.50,"1,413.50",2.00,"219,873.10",9.00,31.40


High vs No Discount: mean net revenue per transaction -26.2%, median -18.4%
Discount given up: $0.70M = 3.73% of list revenue
Transactions with a discount: 54.6%


In [10]:
print("Is the gap more than the price cut itself?")
print(f"  Kruskal, net revenue ~ bucket:    p = {kruskal('revenue_usd', 'bucket'):.1e}")
print(f"  Kruskal, list revenue ~ bucket:   p = {kruskal('list_rev', 'bucket'):.3f}")
print(f"  Kruskal, unit price ~ bucket:     p = {kruskal('unit_price_usd', 'bucket'):.3f}")

print("\nDoes deeper discounting lift basket size or ratings?")
rho, p = stats.spearmanr(df.discount_pct, df.units_sold)
print(f"  Spearman, discount vs units per transaction: rho = {rho:.3f}, p = {p:.3f}")
rated = df.dropna(subset=["customer_rating"])
rho, p = stats.spearmanr(rated.discount_pct, rated.customer_rating)
print(f"  Spearman, discount vs rating:                rho = {rho:.3f}, p = {p:.3f}")

print("\nAre deep (15%) discounts targeted at anyone?")
deep = df.discount_pct.eq(15)
for col in ["category", "sales_channel", "customer_segment", "region"]:
    p = stats.chi2_contingency(pd.crosstab(df[col], deep))[1]
    share = df.groupby(col)["discount_pct"].apply(lambda s: (s == 15).mean() * 100)
    print(f"  15% share by {col:17s}: {share.min():.1f}% to {share.max():.1f}%  (chi-square p = {p:.3f})")

Is the gap more than the price cut itself?
  Kruskal, net revenue ~ bucket:    p = 5.6e-08
  Kruskal, list revenue ~ bucket:   p = 0.501
  Kruskal, unit price ~ bucket:     p = 0.439

Does deeper discounting lift basket size or ratings?
  Spearman, discount vs units per transaction: rho = -0.004, p = 0.680
  Spearman, discount vs rating:                rho = 0.014, p = 0.199

Are deep (15%) discounts targeted at anyone?


  15% share by category         : 8.0% to 9.9%  (chi-square p = 0.393)
  15% share by sales_channel    : 8.3% to 9.9%  (chi-square p = 0.516)
  15% share by customer_segment : 8.6% to 9.6%  (chi-square p = 0.635)
  15% share by region           : 8.1% to 9.8%  (chi-square p = 0.916)


In [11]:
gain = (df.loc[deep, "list_rev"] * 0.05).sum()
break_even = (1 - 0.85 / 0.90) * 100
print(f"Scenario: cap the 15% tier at 10%, demand unchanged")
print(f"  Revenue recovered: ${gain / 1e3:.0f}K = {gain / REV * 100:.2f}% of revenue")
print(f"  Break-even unit loss on the capped tier: {break_even:.1f}%")

Scenario: cap the 15% tier at 10%, demand unchanged
  Revenue recovered: $73K = 0.41% of revenue
  Break-even unit loss on the capped tier: 5.6%


**Takeaway.** Discounts give away $0.70M (3.7% of list revenue). The 26% drop in the High bucket is mostly the 15% price cut itself: the median falls 18%, and list value and unit price do not differ across buckets. Units per transaction are flat at every depth, and deep discounts are spread evenly across every category, channel, segment and region. Capping the 15% tier at 10% is worth about $73K if demand holds.

## 6. Sales channel

Channel means differ by 12.6% from best to worst. Is that gap distinguishable from noise?

In [12]:
chan = df.groupby("sales_channel").agg(transactions=("sale_id", "count"), revenue=("revenue_usd", "sum"),
                                       mean=("revenue_usd", "mean"), median=("revenue_usd", "median"))
chan["revenue_share_%"] = chan.revenue / REV * 100
chan = chan.sort_values("mean", ascending=False)
display(chan.round(1))

top, bottom = chan["mean"].idxmax(), chan["mean"].idxmin()
print(f"Top ({top}) vs bottom ({bottom}) mean gap: {(chan.loc[top, 'mean'] / chan.loc[bottom, 'mean'] - 1) * 100:.1f}%")
print(f"Kruskal-Wallis, revenue ~ channel: p = {kruskal('revenue_usd', 'sales_channel'):.3f}")

,transactions,revenue,mean,median,revenue_share_%
sales_channel,,,,,
Carrier Store,1917,"3,189,039.90","1,663.60",826.80,17.70
Online (Apple.com),1940,"3,121,857.60","1,609.20",829.10,17.30
Third-Party Retailer,1892,"2,987,135.60","1,578.80",848.90,16.60
Apple Store,1914,"2,993,323.20","1,563.90",834.40,16.60
Corporate / B2B,1912,"2,901,069.90","1,517.30",840.30,16.10
Authorized Reseller,1925,"2,843,243.20","1,477.00",819.50,15.80


Top (Carrier Store) vs bottom (Authorized Reseller) mean gap: 12.6%
Kruskal-Wallis, revenue ~ channel: p = 0.630


In [13]:
a = df.loc[df.sales_channel == top, "revenue_usd"].to_numpy()
b = df.loc[df.sales_channel == bottom, "revenue_usd"].to_numpy()
rng = np.random.default_rng(42)
boot = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(5000)]
print(f"Bootstrap 95% CI for the top-bottom gap: ${np.percentile(boot, 2.5):,.0f} to ${np.percentile(boot, 97.5):,.0f}")

mde = (1.96 + 0.84) * df.revenue_usd.std() * np.sqrt(2 / chan.transactions.mean())
print(f"Minimum detectable pairwise gap (80% power): ${mde:,.0f} = {mde / df.revenue_usd.mean() * 100:.0f}% of the mean")

ols = smf.ols("np.log(revenue_usd) ~ C(category) + C(sales_channel)", data=df).fit()
p_channel = anova_lm(ols, typ=2).loc["C(sales_channel)", "PR(>F)"]
print(f"Channel effect after controlling for category (Type II ANOVA on log revenue): p = {p_channel:.3f}")

Bootstrap 95% CI for the top-bottom gap: $-4 to $366
Minimum detectable pairwise gap (80% power): $257 = 16% of the mean
Channel effect after controlling for category (Type II ANOVA on log revenue): p = 0.089


**Takeaway.** The 12.6% gap is not significant (p = 0.63), medians differ by less than 4%, and the bootstrap interval for the gap includes zero. After controlling for category the result is p = 0.089, which is inconclusive. With about 1,900 transactions per channel, only gaps of roughly 16% or more are detectable, so a moderate real difference cannot be ruled out.

## 7. Returns

In [14]:
print(df.return_status.value_counts().to_dict())
returned_rev = df.loc[df.returned, "revenue_usd"].sum()
print(f"Return rate: {df.returned.mean() * 100:.2f}%")
print(f"Exchange rate: {(df.return_status == 'Exchanged').mean() * 100:.2f}%")
print(f"Not kept (returned + exchanged): {(df.return_status != 'Kept').mean() * 100:.2f}%")
print(f"Revenue in returned transactions: ${returned_rev / 1e6:.2f}M = {returned_rev / REV * 100:.1f}% of revenue")

by_cat = df.groupby("category")["returned"].agg(["mean", "size"])
by_cat["return_rate_%"] = by_cat["mean"] * 100
by_cat["ci95_+/-"] = 1.96 * np.sqrt(by_cat["mean"] * (1 - by_cat["mean"]) / by_cat["size"]) * 100
by_cat[["return_rate_%", "ci95_+/-"]].sort_values("return_rate_%", ascending=False).round(2)

{'Kept': 10143, 'Returned': 898, 'Exchanged': 459}
Return rate: 7.81%
Exchange rate: 3.99%
Not kept (returned + exchanged): 11.80%
Revenue in returned transactions: $1.48M = 8.2% of revenue


,return_rate_%,ci95_+/-
category,,
iPad,8.41,1.47
AirPods,8.18,1.65
iPhone,8.13,0.91
Mac,7.90,1.22
Accessories,7.27,0.99
Apple Watch,6.84,1.47


In [15]:
rows = []
for col in ["category", "sales_channel", "region", "customer_segment",
            "customer_age_group", "payment_method", "year", "discount_pct"]:
    rates = df.groupby(col)["returned"].mean() * 100
    p = stats.chi2_contingency(pd.crosstab(df[col], df.returned))[1]
    rows.append({"dimension": col, "lowest_%": rates.min(), "highest_%": rates.max(), "chi2_p": p})
display(pd.DataFrame(rows).round(3))

print(f"Kruskal-Wallis, rating ~ return_status: p = {kruskal('customer_rating', 'return_status'):.3f}")

,dimension,lowest_%,highest_%,chi2_p
0,category,6.84,8.41,0.55
1,sales_channel,7.30,8.62,0.51
2,region,7.28,9.29,0.72
3,customer_segment,7.09,8.39,0.28
4,customer_age_group,6.96,8.52,0.34
5,payment_method,7.13,8.33,0.69
6,year,7.17,8.12,0.21
7,discount_pct,6.43,8.63,0.37


Kruskal-Wallis, rating ~ return_status: p = 0.309


**Takeaway.** 7.8% of transactions are returned (11.8% not kept), tied to $1.48M or 8.2% of revenue. No dimension explains the variation (every p-value is above 0.2), and ratings are unrelated to returns. The pattern is systemic, not specific to a product, channel or customer group.

## 8. Trend and seasonality

In [16]:
year = df.groupby("year").agg(revenue=("revenue_usd", "sum"), transactions=("sale_id", "count"),
                              revenue_per_txn=("revenue_usd", "mean"))
year["yoy_%"] = year.revenue.pct_change() * 100
display(year.round(1))

change = lambda col: (year.loc[2024, col] / year.loc[2022, col] - 1) * 100
print(f"2024 vs 2022: revenue {change('revenue'):+.1f}%, transactions {change('transactions'):+.1f}%, "
      f"revenue per transaction {change('revenue_per_txn'):+.1f}%")

,revenue,transactions,revenue_per_txn,yoy_%
year,,,,
2022,"6,030,524.50",3898,"1,547.10",NaN
2023,"5,786,604.90",3735,"1,549.30",-4.00
2024,"6,218,539.90",3867,"1,608.10",7.50


2024 vs 2022: revenue +3.1%, transactions -0.8%, revenue per transaction +3.9%


In [17]:
days = pd.date_range(df.sale_date.min(), df.sale_date.max())
daily = df.groupby("sale_date")["revenue_usd"].sum().reindex(days, fill_value=0)
p_month = stats.kruskal(*[g.values for _, g in daily.groupby(daily.index.month)]).pvalue
print(f"Kruskal-Wallis, daily revenue ~ month of year: p = {p_month:.3f}")

quarter = df.groupby("quarter")["revenue_usd"].sum()
print(f"Q4 vs the average of the other quarters: {(quarter['Q4'] / quarter.drop('Q4').mean() - 1) * 100:+.1f}%")

Kruskal-Wallis, daily revenue ~ month of year: p = 0.527
Q4 vs the average of the other quarters: +4.0%


**Takeaway.** Revenue is flat: 2024 is 3.1% above 2022 while transactions are 0.8% lower, so growth comes from ticket size. There is no month-of-year effect (p = 0.53) and Q4 is only 4% above the other quarters.

## 9. Customer dimensions

In [18]:
for col in ["customer_segment", "customer_age_group", "payment_method"]:
    print(f"Kruskal-Wallis, revenue per transaction ~ {col}: p = {kruskal('revenue_usd', col):.3f}")
print(f"Kruskal-Wallis, rating ~ category: p = {kruskal('customer_rating', 'category'):.3f}")
print(f"Mean rating: {df.customer_rating.mean():.2f}")

Kruskal-Wallis, revenue per transaction ~ customer_segment: p = 0.775
Kruskal-Wallis, revenue per transaction ~ customer_age_group: p = 0.375


Kruskal-Wallis, revenue per transaction ~ payment_method: p = 0.399
Kruskal-Wallis, rating ~ category: p = 0.853
Mean rating: 4.00


**Takeaway.** Revenue per transaction does not differ by customer segment, age group or payment method, and ratings are about 4.0 everywhere. Customer-level targeting has no support in this dataset.